# Classification of complex networks (Lab 7)

## Imports & Support data



In [57]:
! pip install networkx zstandard -q

In [58]:
from pathlib import Path
import matplotlib.pyplot as plt
from io import BytesIO
import re
import zstandard as zstd
import networkx as nx
from urllib.parse import urlparse
import requests

In [59]:
datasets = {
    "biological": [
        {
            "name": "celegans_interactomes/wi2007",
            "path": "https://networks.skewed.de/net/celegans_interactomes/files/wi2007.xml.zst"
        },
        {
            "name": "collins_yeast",
            "path": "https://networks.skewed.de/net/collins_yeast/files/collins_yeast.xml.zst"
        },
        {
            "name": "malaria_genes/HVR_1",
            "path": "https://networks.skewed.de/net/malaria_genes/files/HVR_1.xml.zst"
        },
    ],
    "social": [
        {
            "name": "netscience",
            "path": "https://networks.skewed.de/net/netscience/files/netscience.xml.zst"
        },
        {
            "name": "arxiv_authors/HepPh",
            "path": "https://networks.skewed.de/net/arxiv_authors/files/HepPh.xml.zst"
        },
        {
            "name": "flickr_groups",
            "path": "https://networks.skewed.de/net/flickr_groups/files/flickr_groups.xml.zst"
        },
    ],
    "technological": [
        {
            "name": "internet_as",
            "path": "https://networks.skewed.de/net/internet_as/files/internet_as.xml.zst"
        },
        {
            "name": "as_skitter",
            "path": "https://networks.skewed.de/net/as_skitter/files/as_skitter.xml.zst"
        },
        {
            "name": "python_dependency",
            "path": "https://networks.skewed.de/net/python_dependency/files/python_dependency.xml.zst"
        },
    ],
}

## Support Functions

### download_xml_zst

Downloads a `.xml.zst` file from a given URL into a local directory, creating the directory if needed.

It validates that the URL points to an `.xml.zst` file, streams the download in chunks, and returns the saved file path as a `Path` object.


In [60]:
def download_xml_zst(url: str, output_dir: str = "data/raw") -> Path:
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    filename = Path(urlparse(url).path).name

    if not filename.endswith(".xml.zst"):
        raise ValueError(f"URL does not point to an .xml.zst file: {filename}")

    file_path = output_path / filename

    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    return file_path

### plot_network

Generates a 2D visualization of a graph using a spring layout for node positioning.

It customizes styling (colors, sizes, background), renders nodes and edges with Matplotlib and NetworkX, displays the plot, and returns the computed node positions.


In [61]:
def plot_network(G: nx.Graph):
    pos = nx.spring_layout(
        G,
        seed=42,
        k=0.01,
        iterations=300,
    )

    fig, ax = plt.subplots(figsize=(10, 10))

    background_color = "#cfcfcf"

    fig.patch.set_facecolor(background_color)
    ax.set_facecolor(background_color)

    nx.draw_networkx_edges(
        G,
        pos,
        ax=ax,
        edge_color="#5a5a5a",
        width=0.4,
        alpha=0.7,
    )

    nx.draw_networkx_nodes(
        G,
        pos,
        ax=ax,
        node_size=22,
        node_color="#b72b2f",
        edgecolors="#8a1f22",
        linewidths=0.3,
        alpha=0.95,
    )

    ax.set_axis_off()
    plt.tight_layout(pad=0)
    plt.show()

    return pos

### read_xml_zst_to_networkx

Reads a local compressed `.xml.zst` GraphML file and converts it into a NetworkX graph.

It validates the file, decompresses it, patches unsupported GraphML attribute types, optionally removes incompatible tags, and can convert the graph to undirected or relabel nodes as integers.


In [62]:
from pathlib import Path
from io import BytesIO
import re
import zstandard as zstd
import networkx as nx


def read_xml_zst_to_networkx(
    file_path: str | Path,
    relabel_to_int: bool = False,
) -> nx.Graph:
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    if file_path.suffixes[-2:] != [".xml", ".zst"]:
        raise ValueError(f"Expected a .xml.zst file, got: {file_path}")

    dctx = zstd.ZstdDecompressor()

    with open(file_path, "rb") as compressed_file:
        with dctx.stream_reader(compressed_file) as reader:
            graphml_data = reader.read()

    graphml_text = graphml_data.decode("utf-8", errors="replace")

    valid_graphml_types = {
        "boolean",
        "int",
        "long",
        "float",
        "double",
        "string",
    }

    def patch_attr_type(match):
        attr_type = match.group(1)
        if attr_type in valid_graphml_types:
            return match.group(0)
        return 'attr.type="string"'

    graphml_text = re.sub(
        r'attr\.type="([^"]+)"',
        patch_attr_type,
        graphml_text,
    )

    graphml_text = re.sub(
        r"<y:.*?</y:.*?>",
        "",
        graphml_text,
        flags=re.DOTALL,
    )

    graph = nx.read_graphml(BytesIO(graphml_text.encode("utf-8")))

    graph = nx.Graph(graph)
    graph.remove_edges_from(nx.selfloop_edges(graph))

    if relabel_to_int:
        graph = nx.convert_node_labels_to_integers(
            graph,
            first_label=0,
            ordering="default",
        )

    return graph

### feature_extractor

Extracts structural metrics from a NetworkX graph and returns them as a dictionary.

It computes node/edge counts, degree moments and variance, clustering, shortest path, assortativity, and the giant component fraction.


In [63]:
def feature_extractor(G:nx.Graph) -> dict[str, float]:
    def moment_of_degree_distribution(G: nx.Graph, m: int) -> float:
        N = len(G)

        if N == 0:
            return 0.0

        total = 0

        for node in G.nodes:
            total += G.degree(node) ** m

        return total / N
    
    def giant_component_fraction(G):
        if G.number_of_nodes() == 0:
            return 0.0
        largest_cc = max(nx.connected_components(G), key=len)
        return len(largest_cc) / G.number_of_nodes()
    
    N = G.number_of_nodes()
    M = G.number_of_edges()

    k1 = moment_of_degree_distribution(G, 1)
    k2 = moment_of_degree_distribution(G, 2)

    variance = k2 - k1**2

    av_cl = nx.average_clustering(G)

    if nx.is_connected(G):
        avg_shortest_path = nx.average_shortest_path_length(G)
    else:
        avg_shortest_path = 0.0

    assortativity = nx.degree_assortativity_coefficient(G)

    return {
        "nodes": N,
        "edges": M,
        "k1": k1,
        "k2": k2,
        "degree_variance": variance,
        "average_clustering": av_cl,
        "average_shortest_path": avg_shortest_path,
        "degree_assortativity": assortativity,
        "giant_component_fraction": giant_component_fraction(G)
    }

### generate_random_networks_from_graph

In [64]:
def generate_random_networks_from_graph(G: nx.Graph) -> dict:
    seed = 42

    G_base = G.to_undirected() if G.is_directed() else G.copy()
    G_base.remove_edges_from(nx.selfloop_edges(G_base))

    n = G_base.number_of_nodes()
    m = G_base.number_of_edges()

    avg_degree = (2 * m) / n
    p = (2 * m) / (n * (n - 1))

    degree_sequence = [d for _, d in G_base.degree()]

    ### Create Erdos Renyi Graph
    G_er = nx.gnm_random_graph(n=n, m=m, seed=seed)

    ### Create Configuration Model Graph
    G_conf = nx.configuration_model(degree_sequence, seed=seed)
    G_conf = nx.Graph(G_conf)
    G_conf.remove_edges_from(nx.selfloop_edges(G_conf))

    ### Create Barabasi Albert Model
    m_ba = max(1, min(round(avg_degree / 2), n - 1))
    G_ba = nx.barabasi_albert_graph(n=n, m=m_ba, seed=seed)

    ### Create Watts Strongatz Graph
    k_ws = max(2, round(avg_degree))
    k_ws = k_ws + 1 if k_ws % 2 != 0 else k_ws
    k_ws = min(k_ws, n - 1)
    k_ws = k_ws - 1 if k_ws % 2 != 0 else k_ws
    G_ws = nx.watts_strogatz_graph(n=n, k=k_ws, p=0.1, seed=seed)


    ### Creates Stochastic Block Model Graph
    sizes = [n // 4] * 4
    sizes[-1] += n % 4
    p_in = min(p * 4, 1.0)
    p_out = p * 0.25
    probs = [
        [p_in if i == j else p_out for j in range(4)]
        for i in range(4)
    ]
    G_sbm = nx.stochastic_block_model(sizes=sizes, p=probs, seed=seed)

    return {
        "erdos_renyi": G_er,
        "configuration_model": G_conf,
        "barabasi_albert": G_ba,
        "watts_strogatz": G_ws,
        "stochastic_block_model": G_sbm,
    }

## Biological Networks



In [65]:
for n in datasets["biological"]:
    temp = download_xml_zst(n["path"])
    n["networks"]  = {
        "original": read_xml_zst_to_networkx(
            temp,
            relabel_to_int=True
        )
    }
print(datasets)

{'biological': [{'name': 'celegans_interactomes/wi2007', 'path': 'https://networks.skewed.de/net/celegans_interactomes/files/wi2007.xml.zst', 'networks': {'original': <networkx.classes.graph.Graph object at 0x7a5651bf4ec0>}}, {'name': 'collins_yeast', 'path': 'https://networks.skewed.de/net/collins_yeast/files/collins_yeast.xml.zst', 'networks': {'original': <networkx.classes.graph.Graph object at 0x7a5651da54c0>}}, {'name': 'malaria_genes/HVR_1', 'path': 'https://networks.skewed.de/net/malaria_genes/files/HVR_1.xml.zst', 'networks': {'original': <networkx.classes.graph.Graph object at 0x7a5650b01c10>}}], 'social': [{'name': 'netscience', 'path': 'https://networks.skewed.de/net/netscience/files/netscience.xml.zst'}, {'name': 'arxiv_authors/HepPh', 'path': 'https://networks.skewed.de/net/arxiv_authors/files/HepPh.xml.zst'}, {'name': 'flickr_groups', 'path': 'https://networks.skewed.de/net/flickr_groups/files/flickr_groups.xml.zst'}], 'technological': [{'name': 'internet_as', 'path': 'ht

In [66]:
for n in datasets["biological"]:
    G_temp = n["networks"]["original"]
    print(
        G_temp
    )

Graph named 'celegans_interactomes (wi2007)' with 1496 nodes and 1714 edges
Graph named 'collins_yeast' with 1622 nodes and 9070 edges
Graph named 'malaria_genes (HVR_1)' with 307 nodes and 2812 edges


In [67]:
for n in datasets["biological"]:
    G_temp = n["networks"]["original"]

    n["networks"].update(
        generate_random_networks_from_graph(G_temp)
    )

In [75]:
features = []
for n in datasets["biological"]:
    for key_ng in n["networks"]:
        tmp = feature_extractor(n["networks"][key_ng])
        tmp.update(
            {
                "type": key_ng,
                "network": n["name"]
            }
        )
        features.append(tmp)

In [ ]:
import pandas as pd
import numpy as np